# Integer Program

Creating the basic framework of an integer program that will optimise the amount of seats won

In [ ]:
import geopandas as gpd
import shapely as sp

## Reading in data

In [ ]:
nsw_data = gpd.read_file("data/20190518/nsw_calculated.geojson")
nsw_data.head()

,id,geometry
0,0,"POLYGON ((151.11574 -33.96985, 151.117 -33.969..."
1,1,"POLYGON ((151.10652 -33.96989, 151.10659 -33.9..."
2,2,"POLYGON ((151.07638 -33.93413, 151.08273 -33.9..."
3,3,"POLYGON ((151.08975 -33.95719, 151.07992 -33.9..."
4,4,"POLYGON ((151.11193 -33.97852, 151.11687 -33.9..."


In [ ]:
ex_poly = nsw_data["geometry"].iloc[0]
ex_poly1 = nsw_data["geometry"].iloc[1]

In [ ]:
adjacency_matrix = [[sp.touches(poly_1, poly_2) 
                     for poly_2 in nsw_data["geometry"]] 
                    for poly_1 in nsw_data["geometry"]]

adjacency_matrix

## Basic integer programming

We have a set of voronoi areas, $v\in V$ and a set of divisions, $d \in D$. A binary variable $a_{rs}$ determines if two regions $r,s\in R$ are adjacent to each other. 
The  binary decision variable $x_{vd}$, $v \in V, d \in D$ determines if region r is included in division d.

Each voronoi polygon can only be selected for one region:

$$
\sum_{d\in D} x_{vd} = 1 \forall  v \in V
$$

Each division must have at least one voronoi polygon

$$
\sum_{v\in V} x_{vd} > 1 \forall  d \in D
$$

# Setting up IP

In [ ]:
from ortools.linear_solver import pywraplp
from ortools.graph.python import linear_sum_assignment

# Create the solver (use CBC for IP/MIP)
solver = pywraplp.Solver.CreateSolver('CBC')
# OR use: solver = pywraplp.Solver.CreateSolver('SCIP')

# Create variables
        # binary variable {0,1}
z = solver.NumVar(0, solver.infinity(), 'z')  # continuous variable

### Define variables

In [ ]:
num_divisions = 10
num_voronoi = len(nsw_data)

In [ ]:

x = {}
for v in range(num_voronoi):
    for d in range(num_divisions):
        x[v, d] = solver.IntVar(0, 1, "")


### Define constraints

In [ ]:
# Assignment constraints
for v in range(num_voronoi):
    model.add(solver.sum(x[v,d] for d in range(num_divisions)) == 1)


for d in range(num_divisions):
    model.add(solver.sum(x[v,d] for v in range(num_voronoi)) > 1)

### Define objective

In [ ]:
objective_terms = []
#starting. withasimpleobjectivefunctiontocheckthatihavetheconstrainsright
for v in range(num_voronoi):
    for d in range(num_divisions):
        objective_terms.append(x[v, d])
        
solver.Minimize(solver.Sum(objective_terms))

In [ ]:
status = solver.Solve()
